# Rust 103: Functions

Chapter 03 of the Rust track. Chapters 01 and 02 covered values and control flow. This chapter introduces **functions**: parameters, return values, tail expressions, tuples, `const fn`, closures, higher-order functions, generics, lifetimes (preview), and diverging functions.

## Who this is written for
You are assumed to be a competent programmer, arriving from Python and C++. The focus is on where Rust differs: explicit return types, tail expressions, ownership in parameters, and the absence of implicit conversions.

## How to read this notebook
- Markdown cells explain; code cells demonstrate. Run each code cell in order.
- Code that intentionally fails is shown as a comment with the compiler error.
- The prose is terse and factual: this is a reference, not a novel.

## The one idea to internalise

Rust functions are **strictly typed and expression-oriented**. The last expression in a body is the return value. Parameters are immutable unless declared `mut`. Functions compose cleanly because their contracts are explicit.

A useful way to read any function in this chapter is from the outside in:

```text
signature → ownership/borrowing → body → return value
```

The signature tells you what the caller is allowed to provide and what the function promises to produce. The body must then satisfy that contract. As the chapter progresses, the signature itself becomes richer: closures add callable behavior, generics add type parameters, lifetimes describe relationships between references, and `Result` makes recoverable failure explicit.


### Contents
- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. Function Basics](#1-function-basics)
- [2. Parameters and Return Values](#2-parameters-and-return-values)
- [3. Tail Expressions](#3-tail-expressions)
- [4. Multiple Returns via Tuples](#4-multiple-returns-via-tuples)
- [5. `const fn`](#5-const-fn)
- [6. Closures and Anonymous Functions](#6-closures-and-anonymous-functions)
- [7. Functions as First-Class Values](#7-functions-as-first-class-values)
- [8. Higher-Order Functions](#8-higher-order-functions)
- [9. Generic Functions](#9-generic-functions)
- [10. Lifetimes in Function Signatures (Preview)](#10-lifetimes-in-function-signatures-preview)
- [11. Why Rust Has No Overloading](#11-why-rust-has-no-overloading)
- [12. Inline Functions and Performance](#12-inline-functions-and-performance)
- [13. Error Handling with `Result`](#13-error-handling-with-result)
- [14. Diverging Functions](#14-diverging-functions)
- [15. Function Pointers vs Closures](#15-function-pointers-vs-closures)
- [16. Cheat Sheet](#16-cheat-sheet)


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the evcxr kernel, which means you are effectively typing inside a permanent `main` function. Functions defined here are globally visible in later cells, and redefinitions replace earlier ones. Items are order-independent, but statements are not. Restarting the kernel clears state and is the correct fix for redefinition errors. Remember that `println!` writes to stdout, while `dbg!` and `eprintln!` write to stderr. Treat each cell as a small experiment: run them in order, and rely on the compiler’s diagnostics when something fails. Functions defined here behave exactly as they would in a real Rust program.

### A useful notebook discipline

Because this is an interactive environment, it is worth distinguishing **learning state** from **program structure**. A later cell may successfully call a function defined many cells earlier, even if a standalone Rust source file would place the definition inside a module or use a different organization.

When experimenting, change one thing at a time: the parameter type, the return type, a semicolon, or the ownership mode. Compiler diagnostics are particularly informative for functions because they tell you which part of the declared contract was violated.


In [ ]:
// 0.1 Smoke test
fn add_one(x: i32) -> i32 { x + 1 }
println!("add_one(5) = {}", add_one(5));


### Reading a function as a contract

A function signature is the boundary between two pieces of code. In:

``` rust
fn add_one(x: i32) -> i32
```

the caller learns three things before looking at the implementation:

1. exactly one `i32` is required;
2. the call produces an `i32`;
3. the implementation cannot silently change that contract without the compiler noticing.

This is an important difference from dynamically typed languages. In Python, a function annotation can document the intended type, but ordinary Python execution does not require the argument to satisfy it. Rust makes the type part of the language semantics.

The explicit contract becomes increasingly valuable as functions grow more sophisticated. Later in this chapter the signature will also describe generic constraints, borrowed references, possible errors, and even non-returning control flow.


## 1. Function Basics

Functions in Rust are declared with `fn`. Parameters require explicit types, and return types follow `->`. Unlike Python, Rust does not allow implicit typing of parameters, and unlike C++ there is no default mutability. This strictness prevents accidental type drift and makes functions predictable. Functions are small contracts: they declare exactly what goes in and what comes out. The compiler enforces these contracts, so you cannot accidentally call a function with the wrong type or forget a return value. This makes Rust functions reliable building blocks for larger programs.


In [ ]:
// 1.1 Simple function
fn greet() {
    println!("Hello from a function");
}
greet();


### Parameters are immutable bindings by default

The `mut` on a parameter controls the **local binding**:

``` rust
fn increment(mut value: i32) -> i32 {
    value += 1;
    value
}
```

It does not mean that the caller's original storage has been modified. That question depends on whether the function owns the value or merely borrows it.

This distinction becomes essential once references are introduced:

- `T` can transfer ownership into the function;
- `&T` borrows without permitting mutation through that reference;
- `&mut T` borrows with permission to mutate the referenced value.

So when reading a function signature, ask two separate questions: **what type enters the function?** and **what ownership/borrowing relationship does the parameter establish?**


## 2. Parameters and Return Values

Parameters are immutable bindings by default. If you need mutability, you must declare `mut` explicitly. This design enforces clarity in APIs and prevents hidden side effects. Return values are explicit. Unlike Python, which can return anything at any time, Rust requires a declared return type. This makes functions predictable and ensures the caller knows exactly what to expect. This strictness is part of Rust’s safety guarantees: every function call is type-checked, and every return value is accounted for. Functions therefore serve as contracts between caller and callee, enforced at compile time.


In [ ]:
// 2.1 Function with parameter and return
fn square(n: i32) -> i32 {
    n * n
}
println!("square(6) = {}", square(6));


### The semicolon can change the return type

The final expression is special because it supplies the value of the enclosing block:

``` rust
fn add(a: i32, b: i32) -> i32 {
    a + b
}
```

Adding a semicolon turns that expression into a statement:

``` rust
fn add(a: i32, b: i32) -> i32 {
    a + b;
}
```

The second body evaluates to `()`, so it no longer satisfies the `i32` return type.

This same rule appears throughout Rust. `if`, `match`, and ordinary `{ ... }` blocks can all produce values. Once you recognize the last expression as the block's value, the syntax of Rust functions becomes much less mysterious.


## 3. Tail Expressions

The last expression in a block is the return value. No `return` keyword is needed unless you want an early exit. This design emphasizes that functions are expressions, not imperative sequences. This makes Rust code read more like mathematics: the function body is a definition, not a series of commands. It also reduces boilerplate and makes the intent clearer. Early returns are still possible with `return`, but the idiomatic style is to let the final expression be the return value. This encourages a functional style where functions are concise and expressive.


In [ ]:
// 3.1 Tail expression
fn cube(n: i32) -> i32 {
    n * n * n
}
println!("cube(3) = {}", cube(3));


### Tuples versus named result types

Tuple returns are excellent when the relationship between the values is obvious and the result is small:

``` rust
let (minimum, maximum) = min_max(3, 7);
```

If the result becomes part of a larger API, however, a struct often communicates intent better:

``` rust
struct Bounds {
    min: i32,
    max: i32,
}
```

With a tuple, the meaning comes from position. With a struct, the meaning comes from a field name. This is an API-design decision rather than merely a stylistic preference.

Tuple destructuring also connects this chapter to Chapter 02: the caller is using a pattern to unpack one compound value into separate bindings.


## 4. Multiple Returns via Tuples

Rust has no multiple return values, but tuples serve the role. This is similar to Python’s ability to return multiple values, but in Rust it is explicit and type-checked. Tuples are lightweight and can be destructured easily. This makes them a convenient way to return related values without creating a struct. For more complex data, prefer structs or enums, but tuples are perfect for small, related values. The compiler enforces the types of each element, so you cannot accidentally mix them up. This makes tuple returns safe and predictable.


In [ ]:
// 4.1 Return a tuple
fn min_max(a: i32, b: i32) -> (i32,i32) {
    if a < b { (a,b) } else { (b,a) }
}
println!("min_max(3,7) = {:?}", min_max(3,7));


### Compile-time evaluation is not the same as optimization

`const fn` is about **where evaluation is permitted**, not about promising faster runtime execution.

For example:

``` rust
const VALUE: i32 = double(21);
```

requires a compile-time value, so Rust evaluates the function in a constant context.

By contrast:

``` rust
let value = double(runtime_input);
```

is ordinary runtime code when `runtime_input` is only known at runtime.

The compiler may optimize the second call anyway, but that is a separate optimization process. Keeping these concepts separate makes `const fn` much easier to reason about: its defining property is compile-time evaluability.


## 5. `const fn`

Rust allows certain functions to be marked as `const fn`, meaning they can be evaluated at compile time. This is particularly useful when defining constants that depend on computation rather than literal values. For example, you might want to compute a constant array length or derive a constant offset. Unlike C++’s `constexpr`, Rust’s `const fn` is more restrictive: only operations guaranteed to be safe and deterministic at compile time are allowed. This ensures that no runtime behavior leaks into compile-time evaluation. The benefit is that you can write expressive code without sacrificing performance, since the compiler precomputes the result. Using `const fn` also improves clarity: readers know the function is pure and side-effect free. As Rust evolves, more operations become allowed in `const fn`, making it a powerful tool for metaprogramming and efficient library design.


In [ ]:
// 5.1 const fn
const fn double(x: i32) -> i32 { x*2 }
const VALUE: i32 = double(21);
println!("VALUE = {}", VALUE);


### Closure capture is the important difference

A closure may depend on variables that are not explicit parameters:

``` rust
let offset = 10;
let add_offset = |x: i32| x + offset;
```

The closure therefore represents more than a function signature; it also carries the relationship with its captured environment.

Rust classifies that relationship through `Fn`, `FnMut`, and `FnOnce`:

- `Fn` — callable without consuming captured state;
- `FnMut` — may mutate captured state;
- `FnOnce` — may consume captured state.

These are traits describing what the closure can do. They are not three different closure syntaxes.


## 6. Closures and Anonymous Functions

Closures are Rust’s version of anonymous functions. They are written using the syntax `|params| expression` and can capture variables from their environment. This makes them similar to Python’s `lambda` or C++’s function objects, but with stricter typing and explicit capture semantics. Closures are powerful because they allow you to write inline behavior without defining a separate named function. They can be stored in variables, passed as arguments, or returned from functions. Rust categorizes closures into three traits depending on how they capture variables: `Fn` for immutable borrow, `FnMut` for mutable borrow, and `FnOnce` for taking ownership. This distinction ensures that closures interact safely with the ownership system. For example, a closure that consumes a variable can only be called once. This explicitness prevents subtle bugs common in other languages. Closures are widely used in iterator adapters like `map`, `filter`, and `fold`, making them central to Rust’s functional programming style.


In [ ]:
// 6.1 Closure example
let add_two = |x: i32| x+2;
println!("add_two(5) = {}", add_two(5));


### Three callable concepts

It is useful to keep these separate:

- a **function item** is the specific named function produced by a `fn` definition;
- a **function pointer** has a type such as `fn(i32) -> i32`;
- a **closure** is an anonymous value that may capture its environment.

A named function can coerce to a function pointer when the signature matches. A capturing closure cannot become an ordinary function pointer because a function pointer has no place to store its captured environment.

This distinction explains why generic APIs often prefer closure traits such as `Fn`: they can accept both ordinary functions and a wider range of closures.


## 7. Functions as First-Class Values

In Rust, functions are first-class values. This means you can assign them to variables, pass them as arguments, and return them from other functions. This property allows you to treat functions like any other data, enabling higher-order programming patterns. For example, you can store a function pointer in a collection or dynamically choose which function to call at runtime. Unlike closures, plain functions cannot capture environment variables, but they are simpler and often more efficient. The type of a function pointer is written as `fn(param_types) -> return_type`. This explicit typing ensures safety and clarity. Treating functions as first-class values makes Rust more expressive and allows you to write generic code that operates on behavior rather than just data. It also bridges imperative and functional styles, giving you flexibility in how you design APIs.


In [ ]:
// 7.1 Assign function to variable
fn add(a: i32, b: i32) -> i32 { a+b }
let f: fn(i32,i32)->i32 = add;
println!("f(2,3) = {}", f(2,3));


### Higher-order functions separate data from behavior

A function such as `apply_twice` is interesting because the input is not only data:

``` rust
apply_twice(plus_one, 5)
```

The integer `5` supplies data, while `plus_one` supplies behavior.

This pattern appears everywhere in Rust's iterator APIs. A traversal can be implemented once while the caller supplies a closure describing the transformation, predicate, or accumulation rule.

The result is composable code: the mechanism handles *how* items are visited, while the supplied function describes *what* should happen to each item.


## 8. Higher-Order Functions

Higher-order functions are functions that take other functions or closures as arguments, or return them as results. This concept is central to functional programming and is well-supported in Rust. For example, you can write a function that applies another function twice, or one that decorates a function with logging. Rust’s standard library makes heavy use of higher-order functions, especially in iterator combinators like `map`, `filter`, and `fold`. These allow you to express complex data transformations concisely and safely. The type system ensures that the functions you pass in have the correct signature, preventing runtime errors. Higher-order functions encourage composition: small, reusable pieces of logic can be combined into larger workflows. This style reduces boilerplate and makes code easier to reason about. Compared to Python, Rust’s higher-order functions are more explicit about types, but this explicitness pays off in safety and performance.


In [ ]:
// 8.1 Pass a function as argument
fn apply_twice(f: fn(i32)->i32, x: i32) -> i32 {
    f(f(x))
}
fn plus_one(n: i32) -> i32 { n+1 }
println!("apply_twice(plus_one, 5) = {}", apply_twice(plus_one, 5));


### Generic does not mean dynamically typed

An unconstrained generic function must be valid for every possible `T`:

``` rust
fn identity<T>(x: T) -> T {
    x
}
```

If the implementation needs an operation that arbitrary `T` does not guarantee, a trait bound supplies the missing contract.

This is a key difference from Python. Python can accept a value and discover at runtime whether an operation exists. Rust instead asks you to state the required capability in the type system.

Monomorphization then lets the compiler specialize generic code for the concrete types actually used, which is why generic abstraction does not inherently imply runtime dispatch or boxing.


## 9. Generic Functions

Rust supports generic functions, which allow you to write code that works with multiple types. Generics are declared using angle brackets, such as `fn identity<T>(x: T) -> T`. This means the function can accept any type and return the same type. Generics are monomorphised at compile time: the compiler generates specialized versions of the function for each type used. This ensures zero-cost abstraction, meaning generic code runs as fast as hand-written type-specific code. You can also add trait bounds to restrict what types are allowed, such as `fn print_debug<T: Debug>(x: T)`. This ensures that only types implementing the `Debug` trait can be passed. Generics make code more reusable and expressive, allowing you to write libraries that work across many types without sacrificing performance. Compared to Python’s dynamic typing, Rust’s generics are explicit and enforced at compile time, preventing type errors before the program runs.


In [ ]:
// 9.1 Generic identity function
fn identity<T>(x: T) -> T { x }
println!("identity(42) = {}", identity(42));
println!("identity('hi') = {}", identity("hi"));


### Lifetimes describe relationships, not time

The `'a` in:

``` rust
fn longest<'a>(x: &'a str, y: &'a str) -> &'a str
```

is not a timestamp and does not extend either string's lifetime.

It says that the returned reference is tied to the lifetime relationship represented by `'a`. Since the function may return either input, the compiler must ensure that the returned reference cannot outlive the data it points to.

Many common functions do not need explicit lifetime syntax because Rust can infer the relationship using lifetime elision rules. Explicit annotations become important when the relationship cannot be inferred unambiguously.

The key mental model is therefore: **lifetimes describe validity relationships between borrows**.


## 10. Lifetimes in Function Signatures (Preview)

Rust’s ownership system ensures memory safety without garbage collection, but references introduce complexity. Lifetimes are annotations that describe how long references remain valid. In function signatures, lifetimes express relationships between input and output references. For example, a function returning one of its arguments must specify that the output reference lives at least as long as the input. This prevents dangling references and enforces safety. Lifetimes are often inferred, but sometimes must be written explicitly. This section provides a preview; later chapters cover lifetimes in depth. Understanding lifetimes is crucial for mastering Rust, as they ensure that borrowed data never outlives its owner. Compared to C++, where dangling pointers are a common bug, Rust’s lifetimes eliminate this class of errors entirely. They may feel verbose at first, but they encode valuable guarantees that make large systems reliable.


In [ ]:
// 10.1 Function with explicit lifetime
fn longest<'a>(x: &'a str, y: &'a str) -> &'a str {
    if x.len() > y.len() { x } else { y }
}
println!("longest = {}", longest("short","longer"));


## 11. Why Rust Has No Overloading

Rust deliberately avoids function overloading by parameter type or count. In languages like C++ or Java, you can define multiple functions with the same name but different signatures. While convenient, this can lead to ambiguity and complex resolution rules. Rust instead encourages clarity: if two functions do different things, they should have different names, or you should use generics and traits to unify behavior. This design choice makes code easier to read and avoids subtle bugs where the wrong overload is chosen. It also simplifies the compiler’s job, since there is only one function definition per name. The result is a language where function resolution is predictable and explicit, and where polymorphism is achieved through traits rather than ad-hoc overloading.


### Overloading versus polymorphism

These are related but different ideas.

**Overloading** means multiple same-named functions are selected according to their signatures. Rust does not use that mechanism.

**Polymorphism** means one operation can work across multiple types. Rust supports this through generics and traits.

For example, instead of creating several overloads, an API can express:

```text
for every T that implements Trait X,
perform operation Y
```

This gives the compiler an explicit capability requirement and avoids a separate overload-resolution system.


## 12. Inline Functions and Performance

Rust allows you to suggest that a function be inlined with the `#[inline]` attribute. Inlining means the compiler replaces a function call with the function’s body, potentially improving performance by eliminating call overhead. However, Rust’s optimizer is already aggressive, and it often inlines functions automatically when beneficial. The attribute is therefore more of a hint than a command. Overusing `#[inline]` can actually harm performance by increasing code size and reducing cache efficiency. The idiomatic approach is to trust the compiler and only use `#[inline]` when you are writing small, frequently called functions in libraries where cross-crate inlining matters. This makes Rust’s performance tuning explicit but restrained, avoiding the pitfalls of premature optimization.


In [ ]:
// 12.1 #[inline] is a hint; the optimizer decides whether to inline.
#[inline]
fn fast_add(a: i32, b: i32) -> i32 { a+b }
println!("fast_add(2,3) = {}", fast_add(2,3));


### `#[inline]` is a hint, not a command

Rust's optimizer can inline functions without an explicit attribute, and `#[inline]` does not force inlining in every situation.

Inlining can help by removing call overhead and exposing the function body to further optimization. It can also increase generated code size. Larger code may have worse instruction-cache behavior, so "more inlining" is not automatically "more speed."

For performance-sensitive code, the practical workflow is:

1. write the clean abstraction;
2. compile with the intended optimization settings;
3. benchmark or profile;
4. intervene only when measurements justify it.

This is particularly relevant in low-latency systems, where tiny changes can matter, but intuition about source code is still an unreliable substitute for generated-code measurements.


## 13. Error Handling with `Result`

Rust does not have exceptions. Instead, it uses the `Result` type to represent computations that may fail. A `Result<T,E>` is either `Ok(T)` for success or `Err(E)` for failure. This makes error handling explicit: every caller must decide how to handle the possibility of failure. The `?` operator provides a concise way to propagate errors upward, reducing boilerplate while keeping control flow visible. Compared to Python, where exceptions can be raised anywhere, Rust’s approach ensures that errors are part of the type system. This prevents surprises and makes code more robust. It also integrates well with pattern matching, allowing you to destructure results and handle success and failure cases directly. The explicitness may feel verbose at first, but it eliminates hidden control flow and makes programs more predictable.


In [ ]:
// 13.1 A Result return makes failure part of the function's signature.
fn parse_num(text: &str) -> Result<i32, std::num::ParseIntError> {
    text.parse::<i32>()
}
println!("parse_num('42') = {:?}", parse_num("42"));
println!("parse_num('oops') = {:?}", parse_num("oops"));


### `Result` makes failure part of the API

A function returning:

``` rust
Result<i32, ParseIntError>
```

does not promise an `i32` unconditionally. It promises a successful value *or* an error.

That changes how callers compose functions. A caller can inspect the two cases with `match`, select one with `if let`, or propagate failure with `?`.

Conceptually, `?` means:

```text
successful value → continue
error             → return that error from the current function
```

So `Result` is not merely an alternative spelling for exceptions. It makes recoverable failure visible in the function's type and keeps the error path explicit in the program's control flow.


## 14. Diverging Functions

A diverging function is one that never returns. Its return type is written as `!`, pronounced “never.” Such functions are used for panics, infinite loops, or process termination. Diverging functions integrate into the type system by acting as placeholders for values that will never exist. For example, a function that panics can be used in a context where a value is required, because the compiler knows control flow will never reach that point. This makes error handling and control flow more consistent. Compared to C++ or Python, where functions that never return are just conventions, Rust encodes this property in the type system, eliminating ambiguity. Diverging functions are rare but important, especially in low-level code or when writing APIs that must signal unrecoverable errors.


In [ ]:
// 14.1 A diverging function is safe to define and never safe to call.
#[allow(dead_code)] // never called: calling it would hang the kernel
fn forever() -> ! {
    loop { println!("running forever"); }
}


### Divergence and expression typing

The `!` type is useful because a diverging expression has no value that will actually be produced.

That lets Rust reason about code such as:

``` rust
let value = if condition {
    42
} else {
    panic!("unreachable");
};
```

The second branch does not need to produce an `i32`: execution never continues from that branch. The compiler knows that the diverging expression can fit into this control-flow position.

This is the same expression-oriented model seen in Chapter 02, now applied to a function whose contract says that normal return is impossible.


## 15. Function Pointers vs Closures

Rust distinguishes between plain function pointers and closures. A function pointer has type `fn(...) -> ...` and refers to a named function. Closures, on the other hand, are anonymous functions that can capture variables from their environment. They implement one of the traits `Fn`, `FnMut`, or `FnOnce`, depending on how they capture. This distinction matters because closures can carry state, while function pointers cannot. For example, a closure can increment a counter defined outside its body, but a function pointer cannot. This makes closures more flexible but also more complex. Function pointers are simpler and often faster, since they do not involve capturing. Understanding the difference helps you choose the right tool: use function pointers for stateless behavior, and closures when you need to capture context. This explicit separation avoids confusion and ensures that performance and semantics are clear.


In [ ]:
// 15.1 A function pointer names code; a closure can carry environment.
fn sum(a: i32, b: i32) -> i32 { a+b }
let f: fn(i32,i32)->i32 = sum;
let closure = |a: i32, b: i32| a+b;
println!("fn pointer: {}", f(2,3));
println!("closure: {}", closure(2,3));


### Choosing between functions and closures

A practical rule of thumb:

- use a **named function** when the operation deserves a stable name and reusable identity;
- use a **closure** when the behavior is local to one operation or needs captured context;
- use a **generic closure bound** when an API should accept both ordinary functions and closures.

For example:

``` rust
let factor = 10;
let scale = |x: i32| x * factor;
```

`scale` cannot be represented by an ordinary function pointer because the call depends on `factor`.

A non-capturing closure has no environment to carry, so it may be convertible to a function pointer. A capturing closure cannot make that conversion without losing the state it needs.


## 16. Cheat Sheet

### Core syntax

- `fn name(params) -> Ret { body }`
- Parameters are immutable unless declared `mut`.
- The tail expression supplies the return value.
- `return value;` is useful for early exits.
- Tuples package small, related return values.
- `const fn` enables compile-time evaluation in permitted contexts.
- Closures use `|x| expr` and may capture their environment.
- Function pointers have types such as `fn(i32) -> i32`.
- Generic functions use type parameters such as `fn f<T>(x: T) -> T`.
- Trait bounds express capabilities required from generic types.
- Lifetimes such as `'a` describe relationships between references.
- Rust has no C++-style function overloading.
- `#[inline]` is an optimization hint, not a guarantee.
- `Result<T, E>` represents success or recoverable failure.
- `?` propagates an error from the current function.
- `-> !` describes a diverging function.
- Function pointers and closures are distinct callable representations.

### What Comes Next
Chapter 04 covers **Strings**: `String` vs `&str`, slicing, UTF-8, and common methods. Strings are central to Rust programming, and understanding them builds on the function concepts introduced here.
